# ⚖️ CourtLLM GRPO Training — Complete Colab-Ready Script

**Runtime:** T4 GPU (free tier) | **Estimated time:** ~45 min for Stage 0-1

This notebook trains an LLM to reduce hallucinations using a multi-agent courtroom environment.

## Cell 1: Install Dependencies

In [ ]:
!pip install -q openenv-core unsloth trl transformers sentence-transformers wikipedia-api httpx matplotlib wandb

## Cell 2: Install CourtLLM Environment from HF Space

In [ ]:
# Replace mishatul with your HuggingFace username
# !pip install git+https://huggingface.co/spaces/mishatul/CourtLLM_OpenEnv

# For local development, clone the repo
!git clone https://huggingface.co/spaces/mishatul/CourtLLM_OpenEnv
!cd CourtLLM_OpenEnv && pip install -e .

## Cell 3: Load Model with Unsloth 4-bit (2x faster, 70% less memory)

In [ ]:
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-7B-Instruct-bnb-4bit",
    max_seq_length=2048,
    load_in_4bit=True,
    dtype=torch.float16,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
    lora_dropout=0.05,
    use_gradient_checkpointing=True,
)

print("Model loaded successfully!")

## Cell 4: Connect to CourtLLM Environment

In [ ]:
from courtllm_env import CourtLLMClient, CourtAction

# Use your deployed HF Space URL
ENV_URL = "https://mishatul-courtllm-openenv.hf.space"
# Or for local testing: ENV_URL = "http://localhost:8000"

# Test connection
with CourtLLMClient(ENV_URL) as client:
    health = client.health()
    print(f"Environment status: {health}")
    
    # Test reset
    obs = client.reset()
    print(f"\nCase ID: {obs.case_id}")
    print(f"Query: {obs.plaintiff_query[:100]}...")
    print(f"Evidence sources: {len(obs.evidence_corpus)}")
    print(f"Flagged claims: {len(obs.flagged_claims)}")

## Cell 5: Helper Functions

In [ ]:
import re
from typing import List

def obs_to_prompt(obs, tokenizer) -> str:
    """Convert observation to Defendant system prompt"""
    
    evidence_str = "\n".join([
        f"[{s['source_id']}] {s['title']}: {s['snippet']}"
        for s in obs.evidence_corpus[:20]  # Limit to first 20 for context
    ])
    
    claims_str = "\n".join([
        f"- {c['claim_id']}: {c['claim_text']} (Reason: {c['suspicion_reason']})"
        for c in obs.flagged_claims
    ])
    
    prompt = f"""You are the Defendant in a legal proceeding about factual accuracy.
Respond to the Plaintiff's query with verifiable evidence for each factual claim.

MANDATORY FORMAT — every claim must use this structure:
<claim>
  <statement>Your factual assertion</statement>
  <source_id>EXACT_SOURCE_ID_FROM_EVIDENCE_CORPUS</source_id>
  <confidence>0.0–1.0</confidence>
</claim>

RULES OF COURT:
1. Only cite source_ids present in the evidence corpus provided.
2. If no source supports a claim, express uncertainty (confidence < 0.4).
3. Contradicting your earlier statements will be penalized.
4. Overconfident wrong claims receive maximum penalty.

Evidence Corpus:
{evidence_str}

Plaintiff's Query: {obs.plaintiff_query}

Flagged Claims:
{claims_str}

Your testimony:"""
    
    return prompt

def parse_defendant_action(completion: str) -> CourtAction:
    """Parse LLM completion into CourtAction"""
    
    # Extract claims using regex
    claim_pattern = r'<claim>.*?<statement>(.*?)</statement>.*?<source_id>(.*?)</source_id>.*?<confidence>(.*?)</confidence>.*?</claim>'
    matches = re.findall(claim_pattern, completion, re.DOTALL)
    
    if not matches:
        # Fallback: treat entire completion as single claim
        return CourtAction(
            action_type="generate_testimony",
            content=completion[:500],
            claim_ids=["claim_000"],
            confidence=0.5,
            source_ids=[]
        )
    
    # Use first claim
    statement, source_id, confidence = matches[0]
    
    try:
        conf = float(confidence.strip())
    except:
        conf = 0.5
    
    return CourtAction(
        action_type="generate_testimony",
        content=statement.strip(),
        claim_ids=["claim_000"],
        confidence=conf,
        source_ids=[source_id.strip()] if source_id.strip() else []
    )

def generate(model, tokenizer, prompt: str, max_length: int = 512) -> str:
    """Generate completion from model"""
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    outputs = model.generate(
        **inputs,
        max_new_tokens=max_length,
        temperature=0.8,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id
    )
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

print("Helper functions loaded!")

## Cell 6: Define Reward Function (OpenEnv + TRL Pattern)

In [ ]:
def courtroom_reward_fn(completions: List[str], prompts: List[str], **kwargs) -> List[float]:
    """
    For each completion, step the environment and return reward.
    This is the exact pattern shown in OpenEnv ceremony deck for TRL.
    """
    rewards = []
    
    with CourtLLMClient(ENV_URL) as env:
        for completion in completions:
            try:
                # Reset for each episode
                obs = env.reset()
                
                # Parse completion into action
                action = parse_defendant_action(completion)
                
                # Step environment
                result = env.step(action)
                
                # Return reward
                rewards.append(result.reward)
            except Exception as e:
                print(f"Error in reward function: {e}")
                rewards.append(-0.5)  # Penalty for errors
    
    return rewards

print("Reward function defined!")

## Cell 7: Build Training Dataset

In [ ]:
from datasets import Dataset

def build_dataset(n_episodes: int = 500, stage: int = 0) -> Dataset:
    """Build prompt dataset from environment resets"""
    prompts = []
    
    with CourtLLMClient(ENV_URL) as env:
        # Set curriculum stage
        env.set_stage(stage)
        
        for i in range(n_episodes):
            obs = env.reset()
            prompt = obs_to_prompt(obs, tokenizer)
            prompts.append({"prompt": prompt})
            
            if (i + 1) % 100 == 0:
                print(f"Generated {i + 1}/{n_episodes} prompts")
    
    return Dataset.from_list(prompts)

# Build training dataset
print("Building training dataset...")
train_data = build_dataset(n_episodes=500, stage=0)
print(f"Dataset size: {len(train_data)}")

## Cell 8: GRPO Training

In [ ]:
from trl import GRPOTrainer, GRPOConfig
import wandb

# Initialize W&B (optional)
wandb.init(project="courtllm-grpo", name="stage0-training")

config = GRPOConfig(
    output_dir="./courtllm_grpo",
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=5e-6,
    num_generations=4,           # G in GRPO — 4 completions per prompt
    max_prompt_length=1024,
    max_completion_length=512,
    temperature=0.8,
    logging_steps=10,
    save_steps=100,
    report_to="wandb",
)

trainer = GRPOTrainer(
    model=model,
    tokenizer=tokenizer,
    reward_funcs=[courtroom_reward_fn],
    args=config,
    train_dataset=train_data,
)

print("Starting GRPO training...")
trainer.train()
print("Training complete!")

## Cell 9: Plot Reward Curves

In [ ]:
import matplotlib.pyplot as plt
import os

# Create outputs directory
os.makedirs("outputs", exist_ok=True)

# Extract training logs
log = trainer.state.log_history
steps = [x["step"] for x in log if "reward" in x]
rewards = [x["reward"] for x in log if "reward" in x]

# Plot reward curve
plt.figure(figsize=(10, 5))
plt.plot(steps, rewards, label="Episode Reward", linewidth=2)
plt.axhline(y=0.65, color='g', linestyle='--', label="Target (0.65)")
plt.xlabel("Training Step", fontsize=12)
plt.ylabel("Average Episode Reward", fontsize=12)
plt.title("CourtLLM GRPO Training — Reward Curve", fontsize=14, fontweight='bold')
plt.legend()
plt.grid(True, alpha=0.3)
plt.savefig("outputs/reward_curve_stage1.png", dpi=150, bbox_inches='tight')
plt.show()

print("Plot saved to outputs/reward_curve_stage1.png")
print("IMPORTANT: Commit this file to your repo!")

## Cell 10: Evaluation — Before/After Conviction Rate

In [ ]:
def eval_conviction_rate(model, tokenizer, env_url: str, n_cases: int = 50) -> float:
    """Evaluate conviction rate on test cases"""
    convictions = 0
    
    with CourtLLMClient(env_url) as env:
        for i in range(n_cases):
            try:
                obs = env.reset()
                prompt = obs_to_prompt(obs, tokenizer)
                completion = generate(model, tokenizer, prompt)
                action = parse_defendant_action(completion)
                result = env.step(action)
                
                if result.reward < 0:
                    convictions += 1
                    
                if (i + 1) % 10 == 0:
                    print(f"Evaluated {i + 1}/{n_cases} cases")
            except Exception as e:
                print(f"Error in case {i}: {e}")
                convictions += 1
    
    return convictions / n_cases

# Load baseline model for comparison
print("Loading baseline model...")
base_model, _ = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-7B-Instruct-bnb-4bit",
    max_seq_length=2048,
    load_in_4bit=True,
)

print("\nEvaluating baseline model...")
baseline_rate = eval_conviction_rate(base_model, tokenizer, ENV_URL, n_cases=50)
print(f"Baseline Conviction Rate: {baseline_rate*100:.1f}%")

print("\nEvaluating trained model...")
trained_rate = eval_conviction_rate(model, tokenizer, ENV_URL, n_cases=50)
print(f"Trained Conviction Rate: {trained_rate*100:.1f}%")

# Plot comparison
labels = ["Baseline\n(untrained)", "CourtLLM\n(GRPO-trained)"]
rates = [baseline_rate * 100, trained_rate * 100]
colors = ["#e74c3c", "#2ecc71"]

plt.figure(figsize=(8, 6))
bars = plt.bar(labels, rates, color=colors, width=0.6)
plt.ylabel("Conviction Rate (%)", fontsize=12)
plt.title("Hallucination Conviction Rate: Before vs After Training", fontsize=14, fontweight='bold')
plt.ylim(0, 100)

for bar, rate in zip(bars, rates):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2,
             f"{rate:.1f}%", ha='center', fontsize=14, fontweight='bold')

plt.grid(True, alpha=0.3, axis='y')
plt.savefig("outputs/conviction_rate_drop.png", dpi=150, bbox_inches='tight')
plt.show()

print("\nPlot saved to outputs/conviction_rate_drop.png")
print("IMPORTANT: Commit this file to your repo!")

## Cell 11: Save Model

In [ ]:
# Save trained model
model.save_pretrained("courtllm_trained")
tokenizer.save_pretrained("courtllm_trained")

print("Model saved to ./courtllm_trained")
print("\nTo upload to HuggingFace:")
print("  from huggingface_hub import HfApi")
print("  api = HfApi()")
print("  api.upload_folder(folder_path='courtllm_trained', repo_id='mishatul/courtllm-7b')")

## Summary

This notebook:
1. ✅ Loaded Qwen2.5-7B with Unsloth 4-bit quantization
2. ✅ Connected to CourtLLM environment
3. ✅ Trained with GRPO using 4-signal reward
4. ✅ Generated reward curves and conviction rate comparison
5. ✅ Saved trained model

**Next steps:**
- Commit `outputs/*.png` files to your repo
- Upload model to HuggingFace
- Write HF blog post with results
- Record <2 min demo video